# 07 – Esercitazione Finale: Orchestrazione Completa

## Obiettivi del Lab (da raggiungere in questa esercitazione)

✅ **Obiettivo 1**: Implementare il sistema con un secondo framework (LangGraph)

✅ **Obiettivo 2**: Aggiungere stato e/o routing condizionale

✅ **Obiettivo 3**: Introdurre un secondo agente o workflow non banale

✅ **Obiettivo 4**: Gestire almeno un failure path

---

## Istruzioni

Questo notebook è il tuo **spazio di lavoro libero**. Dovrai implementare il tuo
caso d'uso (definito nel Project Canvas del Giorno 1) usando i pattern visti oggi.

**Parti dal caso d'uso dell'Assistente Hardware** (già implementato nei notebook 01-03)
e poi adattalo al tuo caso d'uso personale nelle sezioni contrassegnate con `# TODO`.

### Struttura suggerita
1. Definisci lo stato del tuo sistema (TypedDict)
2. Implementa i tool specifici per il tuo dominio
3. Costruisci il grafo LangGraph con almeno 2 agenti
4. Aggiungi routing condizionale
5. Gestisci un failure path (retry, errore, rifiuto)
6. (Bonus) Aggiungi human-in-the-loop

## Parte A: Warm-up — Estendi l'Assistente Hardware

Prima di lavorare sul tuo caso d'uso, completa questi esercizi sull'assistente hardware.
Ogni esercizio aggiunge un tassello che ti servirà nel tuo progetto.

### Warm-up 1: Aggiungere un Nodo di Validazione Input

Aggiungi un nodo `valida_input` all'inizio del grafo che:
- Verifica che `employee_id` esista nel sistema
- Verifica che `richiesta_utente` non sia vuota
- In caso di errore, va direttamente a un nodo `gestisci_errore` che termina il grafo

Deve aggiungere un campo `errore_input` al TypedDict.

In [1]:
# Setup base
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated, Optional

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

llm = ChatOllama(model="llama3.2", temperature=0)
DATA_DIR = Path("data")

with open(DATA_DIR / "employees.json") as f:
    EMPLOYEES_DB = json.load(f)
with open(DATA_DIR / "products.json") as f:
    PRODUCTS_DB = json.load(f)
with open(DATA_DIR / "policy_hardware.md") as f:
    POLICY_TEXT = f.read()

print("✅ Setup completato")

✅ Setup completato


In [2]:
# TODO: Definisci il TypedDict esteso con il campo errore_input
class StatoEsteso(TypedDict):
    richiesta_utente: str
    employee_id: str
    errore_input: str          # vuoto se ok, messaggio di errore se non valido
    risultato_budget: str
    risultato_policy: str
    risultato_catalogo: str
    risposta_finale: str
    log: Annotated[list[str], operator.add]

print("TypedDict definito")

TypedDict definito


In [3]:
# TODO: Implementa il nodo valida_input
def nodo_valida_input(stato: StatoEsteso) -> dict:
    """Valida i parametri di input prima di procedere."""
    # Scrivi qui la logica di validazione
    # ...
    pass


# TODO: Implementa il nodo gestisci_errore
def nodo_gestisci_errore(stato: StatoEsteso) -> dict:
    """Gestisce gli errori di validazione input."""
    # ...
    pass


# TODO: Implementa la conditional edge dopo valida_input
def routing_dopo_validazione(stato: StatoEsteso) -> str:
    """Se c'è un errore, vai a gestisci_errore, altrimenti continua."""
    # ...
    pass

print("Nodi definiti (da implementare)")

Nodi definiti (da implementare)


### Warm-up 2: Ciclo di Retry

L'agente catalogo a volte fallisce (simula un'eccezione casuale).
Implementa un meccanismo di retry che riprova fino a 3 volte prima di arrendersi.

Aggiungi al TypedDict: `retry_catalogo: int` e `catalogo_fallito: bool`.

In [4]:
import random

# TODO: Implementa il nodo catalogo con retry
def nodo_catalogo_con_retry(stato: StatoEsteso) -> dict:
    """
    Cerca prodotti nel catalogo con gestione degli errori.
    Simula un fallimento casuale con probabilità 40% per i primi 2 tentativi.
    """
    retry_count = stato.get("retry_catalogo", 0)
    # Simula fallimento (solo nei primi 2 tentativi)
    if retry_count < 2 and random.random() < 0.4:
        print(f"   ⚠️ Tentativo {retry_count + 1} fallito (simulato)")
        # TODO: aggiorna lo stato con il nuovo retry_count
        pass
    else:
        print(f"   ✅ Catalogo interrogato al tentativo {retry_count + 1}")
        # TODO: esegui la ricerca reale e aggiorna risultato_catalogo
        pass


# TODO: Conditional edge che decide se riprovare o continuare
def routing_catalogo(stato) -> str:
    # ...
    pass

print("Struttura retry definita (da completare)")

Struttura retry definita (da completare)


### Warm-up 3: Assembla il Grafo Completo

Combina la validazione input (Warm-up 1) e il retry catalogo (Warm-up 2) in un
grafo LangGraph completo con tutti i nodi visti nei notebook precedenti.

In [5]:
# TODO: Implementa i nodi mancanti (budget, policy, aggregatore)
# Puoi copiare e adattare dal notebook 02

def nodo_budget(stato) -> dict:
    # ...
    pass

def nodo_policy(stato) -> dict:
    # ...
    pass

def nodo_aggregatore(stato) -> dict:
    # ...
    pass

In [6]:
# TODO: Costruisci il grafo completo
# Struttura attesa:
#
#   START
#     └──► valida_input
#                ├── (errore) ──► gestisci_errore ──► END
#                └── (ok) ──► budget ──► policy ──► catalogo
#                                                        │
#                                              (fallito) ├──► catalogo (retry)
#                                              (ok)      └──► aggregatore ──► END

grafo_completo = StateGraph(StatoEsteso)

# Aggiungi i nodi...
# grafo_completo.add_node(...)

# Aggiungi gli archi...
# grafo_completo.add_edge(...)
# grafo_completo.add_conditional_edges(...)

# app_completo = grafo_completo.compile()
print("Grafo da costruire (completa il codice)")

Grafo da costruire (completa il codice)


---

## Parte B: Il Tuo Caso d'Uso

Ora implementa il tuo sistema agente usando il caso d'uso del tuo Project Canvas.
Segui le istruzioni in ogni cella e sostituisci i placeholder con il tuo codice.

**Ricorda**: l'obiettivo non è la perfezione, ma toccare tutti e 4 gli obiettivi del lab.

### Step 1: Descrivi il Tuo Caso d'Uso

In [7]:
# Compila queste informazioni sul tuo caso d'uso
MIO_CASO_USO = {
    "nome": "[INSERISCI IL NOME DEL TUO SISTEMA]",
    "descrizione": "[DESCRIVI IN 2-3 FRASI COSA FA IL TUO SISTEMA]",
    "input_principale": "[COSA RICEVE IN INPUT? Es: 'una richiesta in linguaggio naturale']",
    "output_atteso": "[COSA PRODUCE IN OUTPUT? Es: 'una raccomandazione strutturata']",
    "agenti_previsti": [
        # Es: {"nome": "AgenteBudget", "responsabilità": "verifica disponibilità economica"}
    ],
    "failure_paths": [
        # Es: "budget insufficiente", "prodotto non disponibile", "policy non rispettata"
    ],
    "human_in_loop": False,  # True se serve approvazione manuale
}

print("Il mio caso d'uso:")
print(json.dumps(MIO_CASO_USO, ensure_ascii=False, indent=2))

Il mio caso d'uso:
{
  "nome": "[INSERISCI IL NOME DEL TUO SISTEMA]",
  "descrizione": "[DESCRIVI IN 2-3 FRASI COSA FA IL TUO SISTEMA]",
  "input_principale": "[COSA RICEVE IN INPUT? Es: 'una richiesta in linguaggio naturale']",
  "output_atteso": "[COSA PRODUCE IN OUTPUT? Es: 'una raccomandazione strutturata']",
  "agenti_previsti": [],
  "failure_paths": [],
  "human_in_loop": false
}


### Step 2: Definisci lo Stato (Obiettivo 2 — Stato strutturato)

In [8]:
# TODO: Definisci il TypedDict per il tuo caso d'uso
# Linee guida:
# - Usa Annotated[list, operator.add] per campi accumulati (log, messaggi)
# - Usa tipi semplici per campi sovrascritti (str, bool, float)
# - Includi un campo per errori/fallimenti (Obiettivo 4)

class MioStato(TypedDict):
    # Input
    input_utente: str
    # TODO: aggiungi altri campi di input specifici per il tuo dominio

    # Stato intermedio
    # TODO: aggiungi i campi per i risultati degli agenti

    # Failure handling (Obiettivo 4)
    errore: str                # vuoto se tutto ok
    retry_count: int

    # Output
    risposta_finale: str
    log: Annotated[list[str], operator.add]

print("Stato definito")

Stato definito


### Step 3: Implementa i Tool (Obiettivo 1 — Secondo framework)

In [9]:
# TODO: Implementa almeno 2 tool specifici per il tuo dominio
# I tool devono interagire con dati reali o simulati

@tool
def mio_tool_principale(parametro: str) -> str:
    """[Descrivi cosa fa questo tool].
    
    Args:
        parametro: [Descrivi il parametro]
    """
    # TODO: implementa la logica
    return f"Risultato per: {parametro}"


@tool
def mio_tool_secondario(input_data: str) -> str:
    """[Descrivi cosa fa questo tool].
    
    Args:
        input_data: [Descrivi il parametro]
    """
    # TODO: implementa la logica
    return f"Analisi di: {input_data}"


MIEI_TOOL = [mio_tool_principale, mio_tool_secondario]
print(f"Tool definiti: {[t.name for t in MIEI_TOOL]}")

Tool definiti: ['mio_tool_principale', 'mio_tool_secondario']


### Step 4: Implementa i Nodi (Obiettivo 3 — Secondo agente)

In [10]:
from langgraph.prebuilt import create_react_agent

# TODO: Crea almeno 2 agenti specializzati

# Agente 1
mio_agente_1 = create_react_agent(
    model=llm,
    tools=[mio_tool_principale],
    prompt="[Scrivi il system prompt per il primo agente]"
)

# Agente 2
mio_agente_2 = create_react_agent(
    model=llm,
    tools=[mio_tool_secondario],
    prompt="[Scrivi il system prompt per il secondo agente]"
)

print("Agenti creati")

Agenti creati


C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_27088\1149954392.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mio_agente_1 = create_react_agent(
C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_27088\1149954392.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mio_agente_2 = create_react_agent(


In [11]:
# TODO: Implementa i nodi del grafo

def nodo_agente_1(stato: MioStato) -> dict:
    """Primo agente specializzato."""
    print("[AGENTE 1] elaborazione...")
    # Invoca mio_agente_1 e aggiorna lo stato
    # ...
    return {"log": ["[AGENTE_1] completato"]}


def nodo_agente_2(stato: MioStato) -> dict:
    """Secondo agente specializzato."""
    print("[AGENTE 2] elaborazione...")
    # Invoca mio_agente_2 e aggiorna lo stato
    # ...
    return {"log": ["[AGENTE_2] completato"]}


def nodo_failure_handler(stato: MioStato) -> dict:
    """Gestisce i failure path (Obiettivo 4)."""
    print("[FAILURE] gestione errore...")
    errore = stato.get("errore", "Errore sconosciuto")
    return {
        "risposta_finale": f"❌ Operazione non completata: {errore}",
        "log": [f"[FAILURE] {errore}"]
    }


def nodo_sintesi_finale(stato: MioStato) -> dict:
    """Sintetizza i risultati in una risposta finale."""
    print("[SINTESI] generazione risposta finale...")
    # TODO: usa l'LLM per sintetizzare i risultati degli agenti
    return {
        "risposta_finale": "[TODO: genera risposta finale]",
        "log": ["[SINTESI] completata"]
    }

print("Nodi definiti")

Nodi definiti


### Step 5: Implementa il Routing Condizionale (Obiettivo 2 — Routing)

In [12]:
# TODO: Implementa almeno una conditional edge

def mio_routing(stato: MioStato) -> str:
    """
    Decidi il prossimo nodo in base allo stato.
    Deve gestire almeno un failure path (Obiettivo 4).
    """
    # Esempio: se c'è un errore, vai al failure handler
    if stato.get("errore"):
        return "fallimento"
    # TODO: aggiungi le tue condizioni di routing
    return "continua"

print("Routing definito")

Routing definito


### Step 6: Assembla e Testa il Grafo

In [13]:
# TODO: Assembla il grafo LangGraph

mio_grafo = StateGraph(MioStato)

# Aggiungi i nodi
mio_grafo.add_node("agente_1", nodo_agente_1)
mio_grafo.add_node("agente_2", nodo_agente_2)
mio_grafo.add_node("fallimento", nodo_failure_handler)
mio_grafo.add_node("sintesi", nodo_sintesi_finale)

# TODO: Definisci il flusso con archi e conditional edges
mio_grafo.add_edge(START, "agente_1")
# mio_grafo.add_conditional_edges("agente_1", mio_routing, {...})
# mio_grafo.add_edge("agente_2", "sintesi")
mio_grafo.add_edge("sintesi", END)
mio_grafo.add_edge("fallimento", END)

# Compila
# mia_app = mio_grafo.compile()
# print("✅ Grafo compilato")

print("Grafo da completare (aggiungi gli archi mancanti)")

Grafo da completare (aggiungi gli archi mancanti)


In [14]:
# TODO: Testa il sistema con almeno 2 casi
# Caso 1: Happy path (tutto funziona)
# Caso 2: Failure path (almeno un errore gestito)

# stato_test_1 = MioStato(
#     input_utente="...",
#     errore="",
#     retry_count=0,
#     risposta_finale="",
#     log=[]
# )
# risultato = mia_app.invoke(stato_test_1)
# print(risultato["risposta_finale"])

print("Test da implementare")

Test da implementare


### Step 7 (Bonus): Human-in-the-Loop

Se il tuo caso d'uso prevede approvazioni manuali, implementa `interrupt()`
in uno dei nodi e testa il flusso di ripresa con `Command(resume=...)`.

In [15]:
# TODO (Bonus): Aggiungi un nodo con interrupt() per approvazione manuale

if MIO_CASO_USO["human_in_loop"]:
    print("Il tuo caso d'uso prevede human-in-the-loop: implementa interrupt()")
    
    def nodo_approvazione(stato: MioStato) -> dict:
        """Sospende il flusso per approvazione manuale."""
        payload = {
            "tipo": "richiesta_approvazione",
            "contesto": stato.get("input_utente"),
            # TODO: aggiungi altri campi rilevanti
        }
        risposta = interrupt(payload)
        # Dopo la ripresa...
        return {
            # TODO: aggiorna lo stato con la decisione
        }
    
    print("Nodo approvazione definito")
else:
    print("Human-in-loop non previsto per questo caso d'uso (imposta human_in_loop=True per attivare)")

Human-in-loop non previsto per questo caso d'uso (imposta human_in_loop=True per attivare)


---

## Checklist di Verifica

Completa questa checklist prima di considerare l'esercitazione terminata.

In [ ]:
def verifica_obiettivi(tuo_stato_classe, tua_app, casi_di_test: list):
    """Verifica automaticamente i 4 obiettivi del lab."""
    print("\n" + "=" * 60)
    print("VERIFICA OBIETTIVI LAB GIORNO 4")
    print("=" * 60)

    risultati = {}

    # Obiettivo 1: Secondo framework
    try:
        from langgraph.graph import StateGraph
        risultati["obj1"] = isinstance(tua_app, type(tua_app))  # è compilato?
        print("✅ Obiettivo 1: Sistema implementato con LangGraph")
    except Exception as e:
        risultati["obj1"] = False
        print(f"❌ Obiettivo 1: {e}")

    # Obiettivo 2: Stato e routing
    import typing
    hints = typing.get_type_hints(tuo_stato_classe)
    has_state = len(hints) >= 3
    risultati["obj2"] = has_state
    print(f"{'✅' if has_state else '❌'} Obiettivo 2: Stato TypedDict con {len(hints)} campi")

    # Obiettivo 3: Secondo agente
    graph_data = tua_app.get_graph().to_json() if hasattr(tua_app, 'get_graph') else {}
    num_nodi = len(graph_data.get("nodes", [])) if isinstance(graph_data, dict) else 0
    has_multi_agent = num_nodi >= 3  # almeno 2 agenti + END
    risultati["obj3"] = has_multi_agent
    print(f"{'✅' if has_multi_agent else '❌'} Obiettivo 3: Grafo con {num_nodi} nodi")

    # Obiettivo 4: Failure path
    failure_tested = False
    for caso in casi_di_test:
        if caso.get("is_failure_test", False):
            failure_tested = True
            break
    risultati["obj4"] = failure_tested
    print(f"{'✅' if failure_tested else '❌'} Obiettivo 4: Failure path testato")

    completati = sum(risultati.values())
    print(f"\n📊 TOTALE: {completati}/4 obiettivi completati")
    if completati == 4:
        print("🎉 Tutti gli obiettivi raggiunti!")
    return risultati


# Esegui la verifica quando il tuo codice è pronto:
# verifica_obiettivi(
#     tuo_stato_classe=MioStato,
#     tua_app=mia_app,
#     casi_di_test=[
#         {"input": "caso normale", "is_failure_test": False},
#         {"input": "caso errore", "is_failure_test": True},
#     ]
# )

print("Funzione di verifica pronta — esegui quando il codice è completato")

Funzione di verifica pronta — esegui quando il codice è completato


: 

## 📝 Note e Osservazioni

Usa questa cella per annotare:
- Sfide incontrate durante l'implementazione
- Decisioni architetturali prese e perché
- Differenze rispetto all'implementazione del Giorno 3
- Domande per il docente

```
Scrivi qui le tue note...
```